# 14 — Embedding ablation: outlier-seed robustness (raw max-sim vs zmean)

A real serving failure: a history of **7 English + 1 foreign-language book** makes raw `max`-sim
flood the feed with that foreign language, regardless of plot — one outlier seed defines a whole
high-scoring same-language neighbourhood. This notebook quantifies it across **TF-IDF · BoW ·
dense embeddings**, and shows that **`zmean` aggregation** (per-seed z-score, then average) fixes
it for every representation — so it's the *aggregation*, not the *encoder*, that stops the hijack.

Metric: share of the top-20 recommendations in the outlier language (here Bulgarian). With 1 of 8
seeds foreign, **12.5% = proportional**; higher = hijack. Feeds §7.2 of the report.

## Kaggle bootstrap
Links `catalog.parquet` + any `embeddings*.npy`. No-op off Kaggle.

In [ ]:
import os, sys, glob
if os.path.exists('/kaggle'):
    REPO = '/kaggle/working/book_recsys'
    if not os.path.exists(REPO):
        !git clone -b master https://github.com/MayaDeneva/book_recsys.git {REPO}
    else:
        !cd {REPO} && git pull origin master
    !pip install -e {REPO} -q
    sys.path.insert(0, REPO); os.chdir(REPO); os.makedirs('artifacts', exist_ok=True)
    for f in ['catalog.parquet', 'embeddings.npy', 'embeddings_0.npy']:
        src = glob.glob(f'/kaggle/input/**/{f}', recursive=True)
        if src and not os.path.exists(f'artifacts/{f}'):
            os.symlink(src[0], f'artifacts/{f}')
    print('artifacts:', sorted(os.listdir('artifacts')))
else:
    print('local repo — expecting artifacts/ in place')

## Setup — catalog, language groups, and the A/B function

In [ ]:
import glob, os, numpy as np, pandas as pd
from book_recsys.models.content.maxsim import MaxSimRecommender
from book_recsys.features.document import build_documents
from book_recsys.features.vectorize import tfidf_matrix, bow_matrix

def find(name):
    for p in (f'artifacts/{name}', f'../artifacts/{name}'):
        if glob.glob(p): return glob.glob(p)[0]
    raise FileNotFoundError(name)

cat = pd.read_parquet(find('catalog.parquet'))
book_ids = cat['book_id'].tolist()
norm = cat['language_code'].fillna('').astype(str).str.lower().str.split('-').str[0].str[:3]
lang_of = dict(zip(book_ids, norm))
english = [b for b, l in zip(book_ids, norm) if l in ('eng', 'en')]

FOREIGN = 'bul'                                   # the outlier language (Bulgarian)
foreign = [b for b, l in zip(book_ids, norm) if l == FOREIGN]
if len(foreign) < 5:                             # fall back to the most common non-English
    FOREIGN = norm[~norm.isin(['eng', 'en', ''])].value_counts().index[0]
    foreign = [b for b, l in zip(book_ids, norm) if l == FOREIGN]
print(f'english={len(english)}  outlier="{FOREIGN}"={len(foreign)}')

TRIALS, K = 40, 20
def foreign_share(matrix, label):
    raw = MaxSimRecommender(book_ids, matrix, agg='max')
    z   = MaxSimRecommender(book_ids, matrix, agg='zmean')
    rng = np.random.default_rng(0)               # same 40 histories for every representation
    r, zz = [], []
    for _ in range(TRIALS):
        seeds = list(rng.choice(english, 7, replace=False)) + [rng.choice(foreign)]
        r.append(np.mean([lang_of[b] == FOREIGN for b in raw.recommend(seeds, K)]))
        zz.append(np.mean([lang_of[b] == FOREIGN for b in z.recommend(seeds, K)]))
    print(f'  {label:28} raw={np.mean(r):5.1%}  zmean={np.mean(zz):5.1%}', flush=True)
    return {'representation': label, f'raw max-sim ({FOREIGN}@{K})': round(float(np.mean(r)), 3),
            f'zmean ({FOREIGN}@{K})': round(float(np.mean(zz)), 3)}

## Run the ablation across representations
TF-IDF and BoW are built from the catalog text; every `embeddings*.npy` in `artifacts/` is included
(e.g. `embeddings.npy` = current multilingual, `embeddings_0.npy` = bge-small-en baseline if kept).

In [ ]:
docs = build_documents(cat)
rows = []
print('building lexical matrices ...', flush=True)
rows.append(foreign_share(tfidf_matrix(docs, max_features=3000)[0], 'TF-IDF'))
rows.append(foreign_share(bow_matrix(docs, 3000)[0], 'BoW'))
for path in sorted(glob.glob('artifacts/embeddings*.npy') + glob.glob('../artifacts/embeddings*.npy')):
    tag = 'embeddings (current)' if path.endswith('embeddings.npy') else os.path.basename(path)
    rows.append(foreign_share(np.load(path), tag))
tbl = pd.DataFrame(rows).set_index('representation')
REPORTS = next((p for p in ('reports', '../reports') if os.path.isdir(p)), 'reports')
os.makedirs(REPORTS, exist_ok=True); tbl.to_csv(f'{REPORTS}/study_embedding_robustness.csv')
print(f'\n12.5% = proportional (1 of 8 seeds foreign); higher = hijack.'); display(tbl)

## Worked example — one history, raw vs zmean (current embeddings)

In [ ]:
emb = np.load(find('embeddings.npy'))
raw = MaxSimRecommender(book_ids, emb, agg='max')
z   = MaxSimRecommender(book_ids, emb, agg='zmean')
rng = np.random.default_rng(1)
seeds = list(rng.choice(english, 7, replace=False)) + [rng.choice(foreign)]
title = dict(zip(cat['book_id'], cat['title']))
print('SEEDS:'); [print('  ', lang_of[b], '·', title.get(b)) for b in seeds]
for name, rec in (('raw max-sim', raw), ('zmean', z)):
    top = rec.recommend(seeds, 10)
    n_for = sum(lang_of[b] == FOREIGN for b in top)
    print(f'\n{name}: {n_for}/10 in "{FOREIGN}"')
    for b in top:
        print('  ', lang_of[b], '·', title.get(b))